# Flujo de Entrenamiento YOLO

Este notebook es una plantilla limpia para entrenar, validar y probar modelos de detección.
It is organized to minimize repeated code and make experiments easier to track.


## 1. Configuración del Entorno

Ejecuta esta celda primero para cargar dependencias, detectar la raíz del proyecto y verificar CUDA.


In [1]:
from __future__ import annotations

import gc
from datetime import date
from pathlib import Path

import torch
import ultralytics
from ultralytics import YOLO
from IPython.display import Image, display

# Resuelve la raíz del proyecto; funciona desde la raíz o desde notebooks/
CWD = Path.cwd().resolve()
PROJECT_ROOT = CWD.parent if CWD.name == "notebooks" else CWD

print(f"Project root: {PROJECT_ROOT}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Torch version: {torch.__version__}")

RUN_ULTRALYTICS_CHECKS = False  # Usa True solo cuando quieras una revisión completa del entorno
if RUN_ULTRALYTICS_CHECKS:
    ultralytics.checks()


Project root: /home/robiotec/Documents/Entrenamientos/Training
CUDA available: True
GPU: NVIDIA GeForce RTX 5090
Torch version: 2.10.0+cu128


## 2. Rutas Compartidas y Valores por Defecto

Keep all reusable paths and defaults in one place.


In [2]:

BASE_MODEL_PATH = PROJECT_ROOT / "model" / "yolo26n.pt"
DATA_ROOT = PROJECT_ROOT / "data"

YAML_CONFIGS = {
    "caja": PROJECT_ROOT / "configs" / "caja.yaml",
    "vetas": PROJECT_ROOT / "configs" / "vetas.yaml",
    "mixed": PROJECT_ROOT / "configs" / "mixto.yaml",
}

# Valores por defecto para entrenamiento. Sobrescribe por experimento solo cuando sea necesario.
DEFAULT_TRAIN_ARGS = {
    "epochs": 150,
    "imgsz": 640,
    "patience": 20,
    "device": 0,
    "warmup_epochs": 3,
    "seed": 42,
    "lrf": 0.1,
    "weight_decay": 0.0001,
    "workers": 8,
    "cache": "disk",
    "plots": True,
}

print(f"Base model: {BASE_MODEL_PATH}")
for name, cfg in YAML_CONFIGS.items():
    print(f"{name:>5}: {cfg}")


Base model: /home/robiotec/Documents/Entrenamientos/Training/model/yolo26n.pt
 caja: /home/robiotec/Documents/Entrenamientos/Training/configs/caja.yaml
vetas: /home/robiotec/Documents/Entrenamientos/Training/configs/vetas.yaml
mixed: /home/robiotec/Documents/Entrenamientos/Training/configs/mixto.yaml


## 3. Funciones de Apoyo

These helpers remove duplicated code for CUDA cleanup, train, validate, and predict.


In [3]:

def clean_cuda(verbose: bool = True) -> None:
    # Libera memoria GPU en caché después de operaciones pesadas.
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        if verbose:
            allocated = torch.cuda.memory_allocated() / 1024**2
            reserved = torch.cuda.memory_reserved() / 1024**2
            print(f"GPU memory - allocated: {allocated:.2f} MB | reserved: {reserved:.2f} MB")


def slug(*parts: str | None) -> str:
    tokens = []
    for part in parts:
        if part is None:
            continue
        text = str(part).strip().lower().replace("-", "_").replace(" ", "_")
        text = "_".join(chunk for chunk in text.split("_") if chunk)
        if text:
            tokens.append(text)
    return "_".join(tokens)


def dataset_base_dir(mine: str, mine_code: str | None = None) -> Path:
    mine_key = mine.strip().upper()
    if mine_key == "BONANZA":
        return DATA_ROOT / "BONANZA"
    if mine_key == "TENGEL":
        if not mine_code:
            raise ValueError("TENGEL requiere mine_code, por ejemplo 'M17' o 'M93'.")
        return DATA_ROOT / "TENGEL" / mine_code.upper()
    raise ValueError(f"Mina no soportada: {mine}")


def model_output_dir(
    mine: str,
    mine_code: str | None,
    model_tag: str,
    version: str,
    run_date: str,
    aug_tag: str = "no_aug",
    model_group: str | None = None,
    model_dir_name: str | None = None,
) -> Path:
    base = dataset_base_dir(mine, mine_code) / "models"
    mine_key = mine.strip().upper()

    # BONANZA agrupa modelos históricos en aug/no_aug/only.
    # Para TENGEL, los modelos viven directo dentro de la carpeta models/ de cada metraje.
    if mine_key == "BONANZA" and model_group:
        base = base / slug(model_group)

    if model_dir_name:
        return base / model_dir_name

    code = "bnz" if mine_key == "BONANZA" else mine_code.lower()
    model_name = slug(code, "model", model_tag, aug_tag, version, run_date)
    return base / model_name


def split_images_dir(mine: str, mine_code: str | None, split_name: str, split: str = "test") -> Path:
    base = dataset_base_dir(mine, mine_code)
    split_root_name = "splits" if mine.strip().upper() == "BONANZA" else "split"
    return base / split_root_name / split_name / split / "images"


def train_model(
    run_dir: Path,
    data_yaml: Path,
    model_path: Path = BASE_MODEL_PATH,
    **overrides,
):
    args = dict(DEFAULT_TRAIN_ARGS)
    args.update(overrides)

    model = YOLO(str(model_path))
    output = model.train(
        data=str(data_yaml),
        project=str(run_dir.parent),
        name=run_dir.name,
        exist_ok=True,
        **args,
    )
    clean_cuda()
    return output


def validate_model(
    model_path: Path,
    data_yaml: Path,
    run_dir: Path,
    split: str = "test",
    conf: float = 0.25,
    **overrides,
):
    model = YOLO(str(model_path))
    output = model.val(
        data=str(data_yaml),
        split=split,
        conf=conf,
        project=str(run_dir.parent),
        name=f"{run_dir.name}__val_{split}",
        save_json=True,
        plots=True,
        exist_ok=True,
        **overrides,
    )
    clean_cuda()
    return output


def predict_images(
    model_path: Path,
    source_path: Path,
    run_dir: Path,
    conf: float = 0.25,
    **overrides,
):
    model = YOLO(str(model_path))
    output = model.predict(
        source=str(source_path),
        conf=conf,
        project=str(run_dir.parent),
        name=f"{run_dir.name}__predict",
        save=True,
        exist_ok=True,
        **overrides,
    )
    clean_cuda()
    return output


## 4. Limpieza Rápida de GPU

Run this anytime you want to free cached CUDA memory with one click.


In [ ]:
clean_cuda()


GPU memory - allocated: 0.00 MB | reserved: 0.00 MB


## 5. Configurar un Entrenamiento

Edit only this cell for each experiment.


In [ ]:
# Fecha usada en el nombre de la carpeta del modelo.
# Si quieres fijar una fecha manualmente, reemplaza `today` por "YYYY-MM-DD" en RUN_CONFIG["run_date"].
today = date.today().isoformat()

# -----------------------------------------------------------------------------
# CONFIGURACIÓN DEL ENTRENAMIENTO
# Edita principalmente este bloque antes de entrenar.
# -----------------------------------------------------------------------------
RUN_CONFIG = {
    # Mina/dataset principal. Valores esperados: "BONANZA" o "TENGEL".
    "mine": "TENGEL",

    # Metraje de TENGEL. Usa "M17", "M93", "M1310", etc. Para BONANZA usa None.
    "mine_code": "M17",

    # Etiqueta descriptiva del modelo. Solo se usa para construir el nombre de salida.
    # Ejemplos: "caja", "veta", "mixed", "only_caja", "only_veta".
    "model_tag": "caja",

    # Config YAML que Ultralytics usa realmente para entrenar.
    # Debe ser una key de YAML_CONFIGS: "caja", "vetas" o "mixed".
    "yaml": "caja",

    # Nombre de la carpeta del split que se utilizará para entrenar el modelo dentro de data/<mina>/<metraje>/split(s)/.
    # Ejemplo TENGEL: data/TENGEL/M17/split/m17_split_white_veins_5cm_rocks_v1
    # Ejemplo BONANZA: data/BONANZA/splits/bnz_split_only_caja_v1
    "split_name": "m17_split_white_veins_5cm_rocks_v1",

    # Versión lógica del modelo. Se usa en el nombre de carpeta si model_dir_name es None.
    "version": "v1",

    # Etiqueta de augmentación para el nombre del modelo.
    # Ejemplos: "no_aug", "aug", "flip_degree_aug", "flip_degree_hsv_aug".
    "aug_tag": "no_aug",

    # Solo para BONANZA: subcarpeta dentro de models. Usa "no_aug", "aug", "only" o None.
    # Para TENGEL normalmente va None porque cada metraje guarda modelos directo en models/.
    "model_group": None,

    # Nombre exacto de carpeta del modelo si quieres forzarlo manualmente.
    # Si es None, se genera automáticamente: <codigo>_model_<model_tag>_<aug_tag>_<version>_<run_date>.
    "model_dir_name": None,

    # Fecha que entra en el nombre generado. Por defecto usa la fecha de hoy.
    "run_date": today,

    # Argumentos enviados a YOLO.train(). Ajusta estos valores por experimento.
    "train_args": {
        # Número de imágenes por batch. Baja este valor si falta memoria GPU.
        "batch": 64,

        # Parada temprana: detiene si no mejora después de N épocas.
        "patience": 30,

        # Tasa de aprendizaje inicial.
        "lr0": 0.01,

        # Augmentación de color: hue, saturation, value.
        "hsv_h": 0.00,
        "hsv_s": 0.0,
        "hsv_v": 0.0,

        # Rotación máxima en grados.
        "degrees": 0,

        # Traslación, escala, shear y perspectiva.
        "translate": 0.0,
        "scale": 0.0,
        "shear": 0.0,
        "perspective": 0.0,

        # Volteos vertical/horizontal.
        "flipud": 0.0,
        "fliplr": 0.5,

        # Augmentaciones compuestas.
        "mosaic": 0.0,
        "mixup": 0.0,
        "copy_paste": 0.0,
        "cutmix": 0.0,

        # Borrado aleatorio y cambio BGR.
        "erasing": 0.0,
        "bgr": 0.0,

        # Entrenamiento multiescala.
        "multi_scale": False,
    },
}

# -----------------------------------------------------------------------------
# RUTAS DERIVADAS AUTOMÁTICAMENTE
# Normalmente no necesitas editar nada debajo de esta línea.
# -----------------------------------------------------------------------------
DATA_YAML = YAML_CONFIGS[RUN_CONFIG["yaml"]]
RUN_DIR = model_output_dir(
    mine=RUN_CONFIG["mine"],
    mine_code=RUN_CONFIG["mine_code"],
    model_tag=RUN_CONFIG["model_tag"],
    version=RUN_CONFIG["version"],
    run_date=RUN_CONFIG["run_date"],
    aug_tag=RUN_CONFIG["aug_tag"],
    model_group=RUN_CONFIG["model_group"],
    model_dir_name=RUN_CONFIG["model_dir_name"],
)
MODEL_TO_VALIDATE = RUN_DIR / "weights" / "best.pt"
PREDICT_SOURCE = split_images_dir(
    RUN_CONFIG["mine"],
    RUN_CONFIG["mine_code"],
    RUN_CONFIG["split_name"],
    split="test",
)

if not DATA_YAML.exists():
    raise FileNotFoundError(f"No encuentro el data.yaml/config: {DATA_YAML}")

print(f"Mine: {RUN_CONFIG['mine']}")
print(f"Mine code: {RUN_CONFIG['mine_code']}")
print(f"Model tag: {RUN_CONFIG['model_tag']}")
print(f"Model dir: {RUN_DIR}")
print(f"YAML config: {DATA_YAML}")
print(f"Split test images: {PREDICT_SOURCE}")


## 6. Entrenar


In [ ]:

train_results = train_model(
    run_dir=RUN_DIR,
    data_yaml=DATA_YAML,
    **RUN_CONFIG["train_args"],
)


## 7. Validar en el Split de Test

Set `MODEL_TO_VALIDATE` to your best checkpoint.


In [ ]:

if not MODEL_TO_VALIDATE.exists():
    raise FileNotFoundError(
        f"No encuentro el modelo entrenado en {MODEL_TO_VALIDATE}. Ejecuta primero la celda de entrenamiento."
    )

val_results = validate_model(
    model_path=MODEL_TO_VALIDATE,
    data_yaml=DATA_YAML,
    run_dir=RUN_DIR,
    split="test",
    conf=0.70,
)


## 8. Predecir en Imágenes Externas (Opcional)

Set `PREDICT_SOURCE` to any folder of images.


In [ ]:

print(f"Prediction source: {PREDICT_SOURCE}")
print(f"Prediction source exists: {PREDICT_SOURCE.exists()}")

# Descomenta para correr predicción sobre las imágenes de test de este split.
# pred_results = predict_images(
#     model_path=MODEL_TO_VALIDATE,
#     source_path=PREDICT_SOURCE,
#     run_dir=RUN_DIR,
#     conf=0.25,
# )


## 9. Comparación Visual Rápida (Opcional)


In [ ]:

# Actualiza estas rutas si quieres comparar matrices de confusión lado a lado.
# from IPython.display import Image, display
#
# train_img = Image(filename=str(RUN_DIR / "confusion_matrix_normalized.png"), width=450)
# val_img = Image(filename=str(RUN_DIR.parent / f"{RUN_DIR.name}__val_test" / "confusion_matrix_normalized.png"), width=450)
#
# try:
#     from ipywidgets import HBox
#     display(HBox([train_img, val_img]))
# except Exception:
#     display(train_img)
#     display(val_img)
